In [28]:
# Persona API Image Extractor
# Extracts all image URLs from Persona inquiries


In [49]:
# ============================================================================
# MAIN PROCESSING: Extract image URLs from Persona inquiries
# ============================================================================
# Fetches inquiry data and extracts all image/photo URLs.
# Output saved to CSV with inquiry_id and list of image URLs.
# ============================================================================

import os
import csv
import time
import json
from typing import Dict, Any, List, Optional

import requests
from dotenv import load_dotenv

# ===============================
# Config
# ===============================
REQUEST_TIMEOUT = 30
SLEEP_S = 0.25  # be kind to API

load_dotenv()
PERSONA_API_KEY = os.getenv("PERSONA_API_KEY")

if not PERSONA_API_KEY:
    raise RuntimeError("Missing PERSONA_API_KEY in .env")

# ===============================
# Persona API
# ===============================
def persona_get_inquiry(inquiry_id: str, include_verifications: bool = True) -> Dict[str, Any]:
    """Retrieve an inquiry.
    
    According to Persona API docs, verifications are included by default,
    but we can explicitly request them with the include parameter.
    """
    from urllib.parse import quote
    encoded_id = quote(inquiry_id, safe='')
    
    url = f"https://api.withpersona.com/api/v1/inquiries/{encoded_id}"
    headers = {
        "Authorization": f"Bearer {PERSONA_API_KEY}",
        "Accept": "application/json",
    }
    
    params = {}
    if include_verifications:
        # Explicitly include verifications to ensure we get full details
        params["include"] = "verifications"
    
    r = requests.get(url, headers=headers, params=params, timeout=REQUEST_TIMEOUT)
    
    if r.status_code == 404:
        raise ValueError(f"Inquiry {inquiry_id} not found (404).")
    
    r.raise_for_status()
    return r.json()

def persona_get_verification(verification_id: str) -> Dict[str, Any]:
    """Retrieve a verification directly from the verification API endpoint.
    
    According to Persona API docs: https://docs.withpersona.com/2023-01-05/api-reference/verifications/retrieve-a-verification
    This endpoint returns full verification details including attributes.
    """
    from urllib.parse import quote
    encoded_id = quote(verification_id, safe='')
    
    url = f"https://api.withpersona.com/api/v1/verifications/{encoded_id}"
    headers = {
        "Authorization": f"Bearer {PERSONA_API_KEY}",
        "Accept": "application/json",
    }
    
    r = requests.get(url, headers=headers, timeout=REQUEST_TIMEOUT)
    
    if r.status_code == 404:
        raise ValueError(f"Verification {verification_id} not found (404).")
    
    r.raise_for_status()
    return r.json()

def get_last_government_id_verification(payload: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    """Get the last (most recent) verification/government-id from inquiry payload."""
    verifications = []
    included = payload.get("included", [])
    
    for item in included:
        if isinstance(item, dict):
            item_type = item.get("type", "")
            if item_type == "verification/government-id":
                verifications.append(item)
    
    # Return the last verification (most recent)
    return verifications[-1] if verifications else None

def extract_front_photo_url_from_last_verification(payload: Dict[str, Any]) -> Optional[str]:
    """Extract front-photo-url from the last (most recent) government-id verification."""
    verification = get_last_government_id_verification(payload)
    
    if verification:
        attributes = verification.get("attributes", {})
        front_photo_url = attributes.get("front-photo-url")
        if isinstance(front_photo_url, str) and front_photo_url:
            return front_photo_url
    
    return None

# ===============================
# Main CSV pipeline
# ===============================
def extract_front_photos_from_csv(input_csv: str, output_csv: str):
    """Read inquiry IDs from CSV and extract front-photo-url from the last verification."""
    fieldnames = ["file_name", "inquiry_id", "front_photo_url", "status"]
    
    # Read input CSV
    with open(input_csv, newline="", encoding="utf-8") as fin:
        reader = csv.DictReader(fin)
        
        # Get available column names (case-insensitive matching)
        if not reader.fieldnames:
            raise ValueError(f"CSV file {input_csv} appears to be empty or invalid.")
        
        # Normalize column names for case-insensitive matching
        # Create mapping: lowercase -> original name
        fieldnames_lower = {name.lower().strip(): name for name in reader.fieldnames}
        
        # Try to find inquiry_id and file_name columns
        inquiry_id_key = None
        file_name_key = None
        
        # Common variations of column names
        inquiry_id_variants = ["inquiry_id", "inquiry id", "inquiryid", "id", "inquiry"]
        file_name_variants = ["file name", "filename", "file_name", "File name"]
        
        for variant in inquiry_id_variants:
            variant_lower = variant.lower().strip()
            if variant_lower in fieldnames_lower:
                inquiry_id_key = fieldnames_lower[variant_lower]
                break
        
        for variant in file_name_variants:
            variant_lower = variant.lower().strip()
            if variant_lower in fieldnames_lower:
                file_name_key = fieldnames_lower[variant_lower]
                break
        
        if not inquiry_id_key:
            print(f"Available columns: {', '.join(reader.fieldnames)}")
            raise ValueError(f"Could not find inquiry_id column. Looking for: {', '.join(inquiry_id_variants)}")
        
        # File name is optional
        if file_name_key:
            print(f"Using columns: '{inquiry_id_key}' for inquiry_id, '{file_name_key}' for file_name\n")
        else:
            print(f"Using column: '{inquiry_id_key}' for inquiry_id (file_name column not found, will use inquiry_id as name)\n")
        
        # Extract inquiries
        inquiries = []
        for row in reader:
            inquiry_id = row.get(inquiry_id_key, "").strip()
            file_name = row.get(file_name_key, "").strip() if file_name_key else inquiry_id
            
            if inquiry_id:
                inquiries.append({"inquiry_id": inquiry_id, "file_name": file_name or inquiry_id})
    
    if not inquiries:
        raise ValueError(f"No inquiries found in {input_csv}. Check that 'inquiry_id' column exists and has data.")
    
    print(f"Found {len(inquiries)} inquiries to process\n")
    
    # Process inquiries and save results
    with open(output_csv, "w", newline="", encoding="utf-8") as fout:
        writer = csv.DictWriter(fout, fieldnames=fieldnames)
        writer.writeheader()
        
        for i, inquiry in enumerate(inquiries, start=1):
            inquiry_id = inquiry["inquiry_id"]
            file_name = inquiry["file_name"]
            
            out = {
                "file_name": file_name,
                "inquiry_id": inquiry_id,
                "front_photo_url": "",
                "status": "OK"
            }
            
            print(f"[{i}/{len(inquiries)}] {file_name} ({inquiry_id})... ", end="", flush=True)
            try:
                payload = persona_get_inquiry(inquiry_id)
                
                front_photo_url = extract_front_photo_url_from_last_verification(payload)
                
                if front_photo_url:
                    out["front_photo_url"] = front_photo_url
                    print(f"✅ Found front photo")
                else:
                    out["status"] = "No government-id verification or no front photo"
                    print(f"⚠️ No front photo found")
                    
            except ValueError as e:
                out["status"] = f"Not found: {e}"
                print(f"❌ {e}")
            except requests.exceptions.HTTPError as e:
                status = getattr(e.response, 'status_code', 'unknown')
                out["status"] = f"HTTP {status}: {str(e)[:100]}"
                print(f"❌ HTTP {status}")
            except Exception as e:
                out["status"] = f"Error: {type(e).__name__}: {str(e)[:100]}"
                print(f"❌ {type(e).__name__}")
            
            writer.writerow(out)
            time.sleep(SLEEP_S)
    
    print(f"\n✅ Done! Front photo URLs saved to: {output_csv}")
    
    # Summary
    import pandas as pd
    summary_df = pd.read_csv(output_csv)
    total = len(summary_df)
    with_photos = len(summary_df[summary_df['front_photo_url'].notna() & (summary_df['front_photo_url'] != '')])
    without_photos = total - with_photos
    
    print(f"\n{'='*60}")
    print(f"SUMMARY")
    print(f"{'='*60}")
    print(f"Total inquiries processed: {total}")
    print(f"Inquiries with front photo: {with_photos} ({with_photos/total*100:.1f}%)")
    print(f"Inquiries without front photo: {without_photos} ({without_photos/total*100:.1f}%)")
    print(f"{'='*60}")

# Run it
INPUT_CSV = "Persona inquiries.csv"
OUTPUT_CSV = "inquiry_front_photos.csv"

try:
    extract_front_photos_from_csv(INPUT_CSV, OUTPUT_CSV)
except FileNotFoundError:
    print(f"❌ File not found: {INPUT_CSV}")
    print(f"\nPlease ensure the CSV file exists with an 'inquiry_id' column.")
except Exception as e:
    print(f"❌ Error: {e}")


Using columns: 'inquiry_id' for inquiry_id, 'File name' for file_name

Found 49 inquiries to process

[1/49] 56170112 (inq_zyJWx11B4uz2Pn9DnQCEGh9f2BpS)... ✅ Found front photo
[2/49] 56170113 (inq_wfRJoBsxBDkb4iVM9j5j8JRLpUsE)... ✅ Found front photo
[3/49] 56172423 (inq_ePnH7pqhTC3iLewPfXxnHy2nRC9h)... ✅ Found front photo
[4/49] 56180354 (inq_rDdu3uPUMCXvCkNKW9h2RmWw17dB)... ✅ Found front photo
[5/49] 56187917 (inq_Gqc1SSrBXiXevWU9ccNf4i3SPnWR)... ✅ Found front photo
[6/49] 56188893 (inq_s5BXVkU8LEMjQs4VCLjZTM6jXdvc)... ✅ Found front photo
[7/49] 56194259 (inq_GZvXfnb1njherThjAgVsgK9DmMP5)... ✅ Found front photo
[8/49] 56194287 (inq_RxNDLDaCLA3FK4YkWXKcAPiLigGb)... ✅ Found front photo
[9/49] 56196714 (inq_wyQRgDJ5CdeKgVxLedc4PS3qXsFR)... ✅ Found front photo
[10/49] 56196715 (inq_TW9eJat1JgZpH1SpN6Kv6kPNJS6q)... ✅ Found front photo
[11/49] 59714483 (inq_oDhZmqLo9UvLPZxWTgMUM8QZokhx)... ✅ Found front photo
[12/49] 59714485 (inq_Ugh1eh1H6RLCyEChS7px6quYbCqt)... ✅ Found front photo
[13/49]

In [ ]:
# ============================================================================
# BULK DOWNLOAD PHOTOS: Download all photos and create ZIP file
# ============================================================================
# Downloads all photos from inquiry_front_photos.csv and creates a ZIP archive
# ============================================================================

from pathlib import Path
from urllib.parse import urlparse
import os
import requests
import pandas as pd
import zipfile
import tempfile
import shutil

def download_photos_to_zip(csv_file: str = "inquiry_front_photos.csv", zip_filename: str = "persona_photos.zip", temp_dir: str = None):
    """Download all photos from the results CSV and create a ZIP file."""
    
    # Create temporary directory for downloads
    if temp_dir:
        temp_path = Path(temp_dir)
        temp_path.mkdir(parents=True, exist_ok=True)
    else:
        temp_path = Path(tempfile.mkdtemp(prefix="persona_photos_"))
    
    # Read CSV
    df = pd.read_csv(csv_file)
    
    # Filter to only rows with photos (handle NaN and empty strings)
    df_with_photos = df[df['front_photo_url'].notna() & (df['front_photo_url'] != '')]
    
    if len(df_with_photos) == 0:
        print("No photos to download.")
        return None
    
    print(f"Downloading {len(df_with_photos)} photos to temporary directory...\n")
    
    downloaded = 0
    failed = 0
    downloaded_files = []
    
    for idx, row in df_with_photos.iterrows():
        inquiry_id = str(row['inquiry_id'])
        file_name = str(row.get('file_name', inquiry_id)) if pd.notna(row.get('file_name')) else inquiry_id
        photo_url = str(row['front_photo_url'])
        
        # Clean file name for filesystem (use only the file_name, no inquiry_id)
        safe_file_name = "".join(c for c in file_name if c.isalnum() or c in (' ', '-', '_')).strip()
        if not safe_file_name:
            safe_file_name = inquiry_id
        
        # Determine file extension from URL or use .jpg as default
        parsed_url = urlparse(photo_url)
        path = parsed_url.path
        if path.lower().endswith(('.jpg', '.jpeg', '.png', '.gif')):
            ext = Path(path).suffix
        else:
            ext = '.jpg'  # Default
        
        # Use only the file name (no inquiry_id suffix)
        output_file = temp_path / f"{safe_file_name}{ext}"
        
        print(f"[{downloaded + failed + 1}/{len(df_with_photos)}] {file_name}... ", end="", flush=True)
        
        try:
            # Download photo
            response = requests.get(photo_url, timeout=30, stream=True)
            response.raise_for_status()
            
            # Save to file
            with open(output_file, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
            
            downloaded_files.append(output_file)
            print(f"✅ Downloaded")
            downloaded += 1
            
        except Exception as e:
            print(f"❌ Failed: {str(e)[:50]}")
            failed += 1
    
    # Create ZIP file
    zip_path = Path(zip_filename)
    print(f"\n📦 Creating ZIP file: {zip_path.name}...")
    
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for photo_file in downloaded_files:
            # Add file to zip with just the filename (not full path)
            zipf.write(photo_file, photo_file.name)
    
    # Clean up temporary directory if we created it
    if not temp_dir:
        shutil.rmtree(temp_path)
        print(f"🧹 Cleaned up temporary directory")
    
    print(f"\n✅ Done! Downloaded {downloaded} photos, {failed} failed")
    print(f"📦 ZIP file created: {zip_path.absolute()}")
    print(f"   File size: {zip_path.stat().st_size / (1024*1024):.2f} MB")
    
    return str(zip_path.absolute())

# Run it
try:
    zip_file = download_photos_to_zip()
    if zip_file:
        print(f"\n💡 To find the ZIP file:")
        print(f"   Open Finder and press Cmd+Shift+G")
        print(f"   Paste this path: {Path(zip_file).parent}")
        print(f"   Or run: open {Path(zip_file).parent}")
except FileNotFoundError:
    print("❌ Results file not found. Run Cell 1 first to process inquiries.")
except Exception as e:
    print(f"❌ Error: {e}")



[1/40] 56170112... ✅ Downloaded
[2/40] 56170113... ✅ Downloaded
[3/40] 56172423... ✅ Downloaded
[4/40] 56180354... ✅ Downloaded
[5/40] 56187917... ✅ Downloaded
[6/40] 56188893... ✅ Downloaded
[7/40] 56194259... ✅ Downloaded
[8/40] 56194287... ✅ Downloaded
[9/40] 56196714... ✅ Downloaded
[10/40] 56196715... ✅ Downloaded
[11/40] 59714483... ✅ Downloaded
[12/40] 59714485... ✅ Downloaded
[13/40] 59714621... ✅ Downloaded
[14/40] 59714797... ✅ Downloaded
[15/40] 59714823... ✅ Downloaded
[16/40] 59715012... ✅ Downloaded
[17/40] 59715479... ✅ Downloaded
[18/40] 59715479... ✅ Downloaded
[19/40] 59715531... ✅ Downloaded
[20/40] 59715715... ✅ Downloaded
[21/40] 59716563... ✅ Downloaded
[22/40] 59716601... ✅ Downloaded
[23/40] 59716954... ✅ Downloaded
[24/40] 59717019... ✅ Downloaded
[25/40] 59717263... ✅ Downloaded
[26/40] 59717660... ✅ Downloaded
[27/40] 59717700... ✅ Downloaded
[28/40] 59718120... ✅ Downloaded
[29/40] 59718182... ✅ Downloaded
[30/40] 59718465... ✅ Downloaded
[31/40] 59718814..

In [53]:
# how many users lacking names in our templates?
# get the images
